In [25]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler

# 1.1 Carga de datos
df = pd.read_csv('titanic_data.csv')

# 1.2 Inspección de dimensiones y tipos
print(f"Dimensiones: {df.shape}")
print("\n--- Conteo de nulos por columna ---")
print(df.isnull().sum())

# 1.3 Vista previa
print("\n--- Primeras 5 filas ---")
print(df.head())

Dimensiones: (889, 15)

--- Conteo de nulos por columna ---
survived         0
pclass           0
sex              0
age            176
sibsp            0
parch            0
fare             0
embarked         2
class            0
who              0
adult_male       0
deck           686
embark_town      2
alive            0
alone            0
dtype: int64

--- Primeras 5 filas ---
   survived  pclass     sex   age  sibsp  parch     fare embarked  class  \
0         0       3    male  22.0      1      0   7.2500        S  Third   
1         1       1  female  38.0      1      0  71.2833        C  First   
2         1       3  female  26.0      0      0   7.9250        S  Third   
3         1       1  female  35.0      1      0  53.1000        S  First   
4         0       3    male  35.0      0      0   8.0500        S  Third   

     who  adult_male deck  embark_town alive  alone  
0    man        True  NaN  Southampton    no  False  
1  woman       False    C    Cherbourg   yes  False

In [28]:
# 2.1 Eliminar duplicados
# Justificación: Registros idénticos no aportan información nueva y sesgan el modelo.
df.drop_duplicates(inplace=True)

# 2.2 Imputación de nulos
# Edad: Usamos la mediana porque es más robusta que el promedio ante valores extremos.
df['age'] = df['age'].fillna(df['age'].median())

# Deck: Como faltan demasiados datos, creamos una categoría nueva "Unknown".
df['deck'] = df['deck'].fillna('Unknown')

# 2.3 Tratamiento de Outliers (Valores atípicos) en 'fare' (Tarifa)
# Justificación: Algunos pasajes costaron 500+, lo que puede distorsionar la escala.
Q1 = df['fare'].quantile(0.25)
Q3 = df['fare'].quantile(0.75)
IQR = Q3 - Q1
limite_superior = Q3 + 1.5 * IQR

# Aplicamos "Capping": los valores por encima del límite se igualan al límite.
df['fare'] = np.where(df['fare'] > limite_superior, limite_superior, df['fare'])
print(f"Limpieza completada. Nuevo máximo de tarifa: {df['fare'].max()}")

Limpieza completada. Nuevo máximo de tarifa: 70.7375


In [29]:
# 3.1 Ingeniería de Features: Tamaño de la familia
# Justificación: En el Titanic, viajar solo o en familia grande influyó en la supervivencia.
df['family_size'] = df['sibsp'] + df['parch'] + 1

# 3.2 Encoding: Convertir 'sex' a números (Mapping)
# Justificación: Los modelos matemáticos no leen texto, leen números.
df['sex_num'] = df['sex'].map({'male': 0, 'female': 1})

# 3.3 Normalización de la Tarifa
# Justificación: Escalar valores entre 0 y 1 ayuda a que el modelo converja más rápido.
scaler = MinMaxScaler()
df['fare_scaled'] = scaler.fit_transform(df[['fare']])

# 3.4 Agregación
# Resumen de supervivencia por clase
print("\n--- Supervivencia promedio por clase ---")
print(df.groupby('class')['survived'].mean())


--- Supervivencia promedio por clase ---
class
First     0.625592
Second    0.509091
Third     0.260000
Name: survived, dtype: float64


In [30]:
# Conteo de nulos por columna
print("Valores nulos por columna:")
print(df.isnull().sum())

# Buscar duplicados
duplicados = df.duplicated().sum()
print(f"\nCantidad de filas duplicadas: {duplicados}")

# Mostrar las filas duplicadas (si existen)
if duplicados > 0:
    print(df[df.duplicated(keep=False)])

Valores nulos por columna:
survived       0
pclass         0
sex            0
age            0
sibsp          0
parch          0
fare           0
embarked       2
class          0
who            0
adult_male     0
deck           0
embark_town    2
alive          0
alone          0
family_size    0
sex_num        0
fare_scaled    0
dtype: int64

Cantidad de filas duplicadas: 0


In [31]:
# Transformación de género a binario
df['sex'] = df['sex'].map({'female': 1, 'male': 0})

# Transformación de la columna 'alive' (opcional, pero útil)
if 'alive' in df.columns:
    df['alive'] = df['alive'].map({'yes': 1, 'no': 0})

# Verificamos los primeros registros
print("--- Columnas transformadas ---")
print(df[['sex', 'alive']].head())

# Verificamos que ya no existan strings
print("\nTipos de datos finales:")
print(df['sex'].dtype)

--- Columnas transformadas ---
   sex  alive
0    0      0
1    1      1
2    1      1
3    1      1
4    0      0

Tipos de datos finales:
int64


In [32]:
df.head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone,family_size,sex_num,fare_scaled
0,0,3,0,22.0,1,0,7.2500,S,Third,man,True,Unknown,Southampton,0,False,2,0,0.102492
1,1,1,1,38.0,1,0,70.7375,C,First,woman,False,C,Cherbourg,1,False,2,1,1.000000
2,1,3,1,26.0,0,0,7.9250,S,Third,woman,False,Unknown,Southampton,1,True,1,1,0.112034
3,1,1,1,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,1,False,2,1,0.750663
4,0,3,0,35.0,0,0,8.0500,S,Third,man,True,Unknown,Southampton,0,True,1,0,0.113801



1. Carga y Exploración
Importamos el archivo titanic_data.csv usando Pandas para identificar el tamaño del dataset, los tipos de datos y, sobre todo, detectar dónde faltaba información (nulos) o dónde había filas repetidas (duplicados).

2. Limpieza de Datos
Duplicados: Eliminamos filas idénticas para no sesgar los resultados.

Nulos: Rellenamos la edad con la mediana y creamos la categoría "Unknown" para la columna de cubierta (deck), evitando así borrar datos valiosos.

Outliers: Ajustamos los precios de los tickets (fare) extremadamente altos para que no distorsionen los promedios.

3. Transformación Avanzada
Encoding: Convertimos columnas a 0 y 1 (numérico) para que los modelos de IA puedan procesarla.

Ingeniería de Features: Creamos la nueva variable family_size sumando parientes, simplificando la estructura del dato.

2. Importancia de la Reproducibilidad
Garantiza que cualquier persona pueda obtener los mismos resultados usando el mismo código. Sin ella, los análisis no son confiables ni auditables.